In [1]:
import pandas as pd
import json

# 1. Resume dataset load karte hain
resume_df = pd.read_csv('../data/raw/Resume.csv')
print("Resume Data Shape:", resume_df.shape)
print("\nResume Columns:", resume_df.columns.tolist())
display(resume_df.head(3))

Resume Data Shape: (2484, 4)

Resume Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
import os

# raw folder ke andar ki files check karte hain
raw_files = os.listdir('../data/raw')
print("Files in raw folder:", raw_files)

# Naukri wali file ka exact naam dhoondh kar load karte hain
job_file = [f for f in raw_files if 'naukri' in f][0]
jobs_df = pd.read_json(f'../data/raw/{job_file}', lines=True)

print("\nJobs Data Shape:", jobs_df.shape)
print("\nJobs Columns:", jobs_df.columns.tolist())
display(jobs_df.head(2))

Files in raw folder: ['marketing_sample_for_naukri_com-jobs_listing__20200101_20200331__30k_data.ldjson', 'Resume.csv']

Jobs Data Shape: (30060, 35)

Jobs Columns: ['uniq_id', 'crawl_timestamp', 'url', 'job_title', 'category', 'company_name', 'city', 'country', 'post_date', 'job_description', 'job_type', 'job_board', 'geo', 'job_post_lang', 'valid_through', 'inferred_iso2_lang_code', 'inferred_iso3_lang_code', 'site_name', 'html_job_description', 'domain', 'postdate_yyyymmdd', 'has_expired', 'last_expiry_check_date', 'latest_expiry_check_date', 'duplicate_status', 'postdate_in_indexname_format', 'inferred_city', 'inferred_state', 'inferred_country', 'fitness_score', 'salary_offered', 'contact_email', 'inferred_salary_currency', 'inferred_salary_from', 'inferred_salary_to']


,uniq_id,crawl_timestamp,url,job_title,category,company_name,city,country,post_date,job_description,...,postdate_in_indexname_format,inferred_city,inferred_state,inferred_country,fitness_score,salary_offered,contact_email,inferred_salary_currency,inferred_salary_from,inferred_salary_to
0,637ff02389d6304fafbbae2cc6d44ca0,2020-01-25 13:02:17 +0000,https://www.naukri.com/job-listings-Business-D...,Business Development Executive,Sales,Ambey Recruitment Services,Faridabad,India,2019-09-18,Job Description\r&lt;br&gt;\r&lt;br&gt;We Requ...,...,2019.09.14,,,India,8,NaN,NaN,NaN,NaN,NaN
1,df5d62b47454b396d3e9be489e2d3d9a,2020-01-30 01:41:28 +0000,https://www.naukri.com/job-listings-BPO-Hindi-...,BPO Hindi Call Center ( CCE ) BPO JOB Fresher ...,ITES,Resource Solution,Delhi,India,2020-01-12,Excellent Opportunity For You To Work With Ind...,...,2020.01.07,Delhi,Delhi,India,10,NaN,NaN,NaN,NaN,NaN


In [4]:
import re

# 1. Text cleaning function (URLs, emails, punctuation, extra spaces hatayenge)
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+\s*', ' ', text)       # links hataye
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text) # emails hataye
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text) # punctuation
    text = re.sub(r'\s+', ' ', text).strip()      # extra spaces hataye
    return text

# 2. Resumes par cleaning apply karte hain
resume_df['clean_resume'] = resume_df['Resume_str'].apply(clean_text)

# 3. Sample check karte hain (Original vs Cleaned)
print("Original Text Sample:")
print(resume_df['Resume_str'].iloc[0][:150])
print("\nCleaned Text Sample:")
print(resume_df['clean_resume'].iloc[0][:150])

<>:10: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<>:10: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
C:\Users\Samsung\AppData\Local\Temp\ipykernel_3972\2085426721.py:10: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
  text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text) # punctuation


Original Text Sample:
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in H

Cleaned Text Sample:
hr administrator marketing associate hr administrator summary dedicated customer service manager with 15 years of experience in hospitality and custom


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Feature (X) aur Target Label (y)
X = resume_df['clean_resume']
y = resume_df['Category']

# 2. Train-Test Split (80% training, 20% testing with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# 4. Model Training (Multinomial Naive Bayes)
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

# 5. Evaluation
y_pred = nb_model.predict(X_test_vec)
print("Model Baseline Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%\n")
print("Classification Report Summary:")
print(classification_report(y_test, y_pred, digits=2, zero_division=0))

Model Baseline Accuracy: 54.93 %

Classification Report Summary:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.50      0.88      0.64        24
              ADVOCATE       0.37      0.42      0.39        24
           AGRICULTURE       1.00      0.08      0.14        13
               APPAREL       1.00      0.05      0.10        19
                  ARTS       0.33      0.05      0.08        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.79      0.62      0.70        24
               BANKING       0.88      0.65      0.75        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.37      0.79      0.51        24
                  CHEF       0.78      0.75      0.77        24
          CONSTRUCTION       0.59      0.59      0.59        22
            CONSULTANT       1.00      0.04      0.08        23
              DESIGNER       0.75     

In [6]:
from sklearn.svm import LinearSVC

# Linear SVM model train karte hain
svm_model = LinearSVC(random_state=42, C=1.0)
svm_model.fit(X_train_vec, y_train)

# Accuracy aur Evaluation
y_pred_svm = svm_model.predict(X_test_vec)
print("Linear SVM Accuracy:", round(accuracy_score(y_test, y_pred_svm) * 100, 2), "%\n")
print(classification_report(y_test, y_pred_svm, digits=2, zero_division=0))

Linear SVM Accuracy: 70.22 %

                        precision    recall  f1-score   support

            ACCOUNTANT       0.72      0.88      0.79        24
              ADVOCATE       0.77      0.71      0.74        24
           AGRICULTURE       0.75      0.46      0.57        13
               APPAREL       0.62      0.26      0.37        19
                  ARTS       0.55      0.29      0.38        21
            AUTOMOBILE       1.00      0.43      0.60         7
              AVIATION       0.86      0.79      0.83        24
               BANKING       0.83      0.83      0.83        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.56      0.62      0.59        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.79      0.86      0.83        22
            CONSULTANT       0.69      0.39      0.50        23
              DESIGNER       0.86      0.90      0.88        21
         

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TF-IDF me unigram + bigram (ngram_range=(1,2)) aur 10,000 features use karenge
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)),
    ('clf', LinearSVC(random_state=42, C=0.5))
])

# Model train karte hain
pipeline.fit(X_train, y_train)

# Evaluate
y_pred_tuned = pipeline.predict(X_test)
print("Optimized Pipeline Accuracy:", round(accuracy_score(y_test, y_pred_tuned) * 100, 2), "%\n")
print(classification_report(y_test, y_pred_tuned, digits=2, zero_division=0))

Optimized Pipeline Accuracy: 70.22 %

                        precision    recall  f1-score   support

            ACCOUNTANT       0.70      0.88      0.78        24
              ADVOCATE       0.57      0.54      0.55        24
           AGRICULTURE       0.88      0.54      0.67        13
               APPAREL       0.60      0.32      0.41        19
                  ARTS       0.73      0.38      0.50        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.83      0.79      0.81        24
               BANKING       0.85      0.74      0.79        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.62      0.83      0.71        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.76      0.86      0.81        22
            CONSULTANT       0.86      0.26      0.40        23
              DESIGNER       0.83      0.90      0.86        21
 

In [8]:
import joblib

# 1. Trained Classifier Pipeline ko models folder me save karte hain
joblib.dump(pipeline, '../models/resume_classifier.pkl')
print("Model saved to models/resume_classifier.pkl")

# 2. Quick Test: Ek custom sample resume text par prediction check karte hain
sample_resume = ["Experienced Python developer proficient in Machine Learning, Pandas, Scikit-learn, and building backend APIs."]
pred = pipeline.predict(sample_resume)
print("Sample Resume Prediction:", pred[0])

Model saved to models/resume_classifier.pkl
Sample Resume Prediction: ENGINEERING


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Jobs data me se zaroori columns clean aur filter karte hain
sample_jobs = jobs_df[['job_title', 'company_name', 'job_description', 'city']].dropna().copy()
sample_jobs['clean_desc'] = sample_jobs['job_description'].apply(clean_text)

# Speed aur memory ke liye pehle 2,000 jobs par TF-IDF fit karte hain
sample_jobs = sample_jobs.head(2000).reset_index(drop=True)
job_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
job_tfidf_matrix = job_vectorizer.fit_transform(sample_jobs['clean_desc'])

# 2. Recommender function
def recommend_jobs(resume_text, top_n=5):
    cleaned = clean_text(resume_text)
    user_vec = job_vectorizer.transform([cleaned])
    sim_scores = cosine_similarity(user_vec, job_tfidf_matrix).flatten()
    
    top_indices = sim_scores.argsort()[-top_n:][::-1]
    results = sample_jobs.iloc[top_indices][['job_title', 'company_name', 'city']].copy()
    results['Match_Score_%'] = (sim_scores[top_indices] * 100).round(2)
    return results

# 3. Test recommendation on sample resume
recommend_jobs(sample_resume[0], top_n=5)

,job_title,company_name,city,Match_Score_%
1333,Data Scientist,Lymbyc,Bengaluru,23.53
140,Platform Development - Python Developer,Golden Opportunities Pvt Ltd.,Bengaluru,21.96
1842,Data Scientist,Zest HR Services.,Mumbai Suburbs,21.66
400,Statistical Researcher_communication Designer_...,Aditech IT and Consultancy Services Pvt Ltd,Pune,19.73
5,PHP Developer (symfony),TRooTech Business Solutions,Ahmedabad,16.06
